In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from datasets import Dataset
import os
from dotenv import load_dotenv
from BaselineModel import BaselineModel
from constant import *
from SimpleOutputLabelConverter import SimpleOutputLabelConverter


/home/cs/grad/islams32/dev/project/academic/technical-debt/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/cs/grad/islams32/dev/project/academic/technical-debt/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce GTX 1080 which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  warnings.warn(
/home/cs/grad/islams32/dev/project/academic/technical-debt/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
/home/cs/grad/islams32/dev/

In [2]:
load_dotenv()
simple_output_label_converter = SimpleOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)

In [3]:
import shutil
BASE_MAT_DIRECTORY = os.getenv('BASE_MAT_DIRECTORY')
MAT_NEW_INPUT_DIRECTORY = f'{BASE_MAT_DIRECTORY}/trained/input'
MAT_NEW_OUTPUT_DIRECTORY = f'{BASE_MAT_DIRECTORY}/trained/output'
shutil.copytree('../config/baseline/dic', MAT_NEW_INPUT_DIRECTORY + '/dic', dirs_exist_ok=True)
os.makedirs(os.path.join(MAT_NEW_INPUT_DIRECTORY, 'origin'), exist_ok=True)
os.makedirs(MAT_NEW_OUTPUT_DIRECTORY, exist_ok=True)
for kv in [{'train': detect_train_df}, {'test': detect_test_df}, {'merged': pd.concat([detect_train_df.assign(project='train'), detect_test_df.assign(project='test')])}]:
    for df_name, df in kv.items():
        if df_name == 'merged':
            df['text'].str.replace('\n', '\t').to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/comments', index=False, header=False)
            df['label'].str.lower().map({'yes': 'SATD', 'no': 'WITHOUT_CLASSIFICATION'}).to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/labels', index=False, header=False)
            df['project'].to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/projects', index=False, header=False)
        else:
            df['text'].str.replace('\n', '\t').to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/data--{df_name}.txt', index=False, header=False)
            df['label'].str.lower().map({'yes': 'positive', 'no': 'negative'}).to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/label--{df_name}.txt', index=False, header=False)




MAT_PRETRAINED_INPUT_DIRECTORY = f'{BASE_MAT_DIRECTORY}/pretrained/input'
MAT_PRETRAINED_OUTPUT_DIRECTORY = f'{BASE_MAT_DIRECTORY}/pretrained/output'
os.makedirs(MAT_PRETRAINED_OUTPUT_DIRECTORY, exist_ok=True)
shutil.copytree('../config/baseline', MAT_PRETRAINED_INPUT_DIRECTORY, dirs_exist_ok=True)

detect_test_df['text'].str.replace('\n', '\t').to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/data--test.txt', index=False, header=False)
detect_test_df['label'].str.lower().map({'yes': 'positive', 'no': 'negative'}).to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/label--test.txt', index=False, header=False)
df1 = pd.DataFrame(open('../config/baseline/origin/data--train.txt').read().splitlines(), columns=["text"])
df2 = detect_test_df[['text']]
pd.concat([df1, df2])['text'].str.replace('\n', '\t').to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/comments', index=False, header=False)

df1 = pd.read_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/label--train.txt', header=None, names=['label']).assign(project='train')
df2 = pd.read_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/label--test.txt', header=None, names=['label']).assign(project='test')
df = pd.concat([df1, df2])
df['label'].map({'positive': 'SATD', 'negative': 'WITHOUT_CLASSIFICATION'}).to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/labels', columns=['label'], index=False, header=False)

df.to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/projects', columns=['project'], index=False, header=False)



# Potdar Pattern

In [5]:
pattern_model = BaselineModel('detect', f'pretrained-potdar-Pattern', simple_output_label_converter)
pattern_model.fit(detect_train_dataset)
pattern_model.predict(detect_test_dataset, DETECT_DATASET_NAME)

detect with pretrained-potdar-Pattern
True
Running model Pattern in MTO
Preparing data for Pattern
Pattern prediction finished!
Method: Pattern
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
233, 5573, 78, 112431, 0.749, 0.040, 0.076, 0.934, 14.267
4, 109, 1, 6416, 0.800, 0.035, 0.068, 0.978, 45.230

Test Result:
              precision    recall  f1-score   support

          no      0.983     1.000     0.992      6418
         yes      0.800     0.035     0.068       113

    accuracy                          0.983      6531
   macro avg      0.892     0.518     0.530      6531
weighted avg      0.980     0.983     0.976      6531



'../cache/output/snapshot/September 15, 2025, 14:37:22$detect_pretrained-potdar-Pattern-pretrained-potdar-Pattern.csv'

In [6]:
trained_pattern_model = BaselineModel('detect', f'trained-potdar-Pattern', simple_output_label_converter)
trained_pattern_model.fit(detect_train_dataset)
trained_pattern_model.predict(detect_test_dataset, DETECT_DATASET_NAME)

detect with trained-potdar-Pattern
True
Running model Pattern in MTO
Preparing data for Pattern
Pattern prediction finished!
Method: Pattern
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
13, 390, 7, 25613, 0.650, 0.032, 0.061, 0.976, 40.973
4, 109, 1, 6416, 0.800, 0.035, 0.068, 0.978, 45.230

Test Result:
              precision    recall  f1-score   support

          no      0.983     1.000     0.992      6418
         yes      0.800     0.035     0.068       113

    accuracy                          0.983      6531
   macro avg      0.892     0.518     0.530      6531
weighted avg      0.980     0.983     0.976      6531



'../cache/output/snapshot/September 15, 2025, 14:37:32$detect_trained-potdar-Pattern-trained-potdar-Pattern.csv'

# Text Mining

In [ ]:
pretrained_tm_model = BaselineModel('detect', f'pretrained-TM', simple_output_label_converter)
pretrained_tm_model.fit(detect_train_dataset)
pretrained_tm_model.predict(detect_test_dataset, DETECT_DATASET_NAME)

In [ ]:
trained_tm_model = BaselineModel('detect', f'trained-TM', simple_output_label_converter)
trained_tm_model.fit(detect_train_dataset)
trained_tm_model.predict(detect_test_dataset, DETECT_DATASET_NAME)

# NLP

In [ ]:
pretrained_nlp_model = BaselineModel('detect', f'pretrained-NLP', simple_output_label_converter)
pretrained_nlp_model.fit(detect_train_dataset)
pretrained_nlp_model.predict(detect_test_dataset, DETECT_DATASET_NAME)

In [ ]:
trained_nlp_model = BaselineModel('detect', f'trained-NLP', simple_output_label_converter)
trained_nlp_model.fit(detect_train_dataset)
trained_nlp_model.predict(detect_test_dataset, DETECT_DATASET_NAME)

# MAT

In [ ]:
pretrained_mat_model = BaselineModel('detect', f'pretrained-MAT', simple_output_label_converter)
pretrained_mat_model.fit(detect_train_dataset)
pretrained_mat_model.predict(detect_test_dataset, DETECT_DATASET_NAME)

In [ ]:
trained_mat_model = BaselineModel('detect', f'trained-MAT', simple_output_label_converter)
trained_mat_model.fit(detect_train_dataset)
trained_mat_model.predict(detect_test_dataset, DETECT_DATASET_NAME)